# 05 — MuJoCo Simulation

Physics simulation with the MuJoCo backend. The `Simulation` class is both
a `SimEngine` implementation and a Strands `AgentTool`.

**Requires**: `pip install "strands-robots[sim-mujoco]"`

In [10]:
from strands_robots.simulation import create_simulation, list_backends

print(f"backends: {list_backends()}")

backends: ['mj', 'mjc', 'mjx', 'mujoco']


## World Lifecycle

In [11]:
sim = create_simulation("mujoco")  # or "mj", "mjc", "mjx" — all aliases

result = sim.create_world(
    timestep=0.002,        # 500 Hz physics
    gravity=[0, 0, -9.81],
    ground_plane=True,
)
print(result["content"][0]["text"])

🌍 Simulation world created
⚙️ Timestep: 0.002s (500Hz physics)
🌐 Gravity: [0, 0, -9.81]
📷 Default camera ready
🤖 Robot models: 62 available
💡 Add robots: action='add_robot' (urdf_path or data_config)
💡 Add objects: action='add_object'
💡 List URDFs: action='list_urdfs'


## Adding Robots

Resolved from the model registry — checks local assets, then Menagerie.

In [12]:
result = sim.add_robot(
    name="so100",
    data_config="so100",
    position=[0, 0, 0],
)
print(result["content"][0]["text"])

🤖 Robot 'so100' added to simulation
📁 Source: data_config='so100' → scene.xml
📍 Position: [0, 0, 0]
🔩 Joints: 6 (Rotation, Pitch, Elbow, Wrist_Pitch, Wrist_Roll, Jaw)
⚡ Actuators: 6
📷 Cameras: ['default']
💡 Run policy: action='run_policy', robot_name='so100'


## Adding Objects

In [13]:
sim.add_object("red_cube", shape="box", position=[0.3, 0, 0.05],
               size=[0.03, 0.03, 0.03], color=[1, 0, 0, 1], mass=0.05)
sim.add_object("blue_sphere", shape="sphere", position=[0.2, 0.1, 0.05],
               size=[0.02], color=[0, 0, 1, 1], mass=0.03)
sim.add_object("table", shape="box", position=[0.3, 0, -0.02],
               size=[0.3, 0.3, 0.02], color=[0.6, 0.4, 0.2, 1], is_static=True)
print("objects added ✅")

objects added ✅


## Stepping & Observation

In [14]:
sim.step(n_steps=100)

obs = sim.get_observation("so100")

# Separate scalar joints from image arrays
joints = {k: v for k, v in obs.items() if not hasattr(v, 'shape') or getattr(v, 'ndim', 0) < 2}
images = {k: v for k, v in obs.items() if hasattr(v, 'shape') and getattr(v, 'ndim', 0) >= 2}

print("joints:")
for k, v in joints.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print(f"images: { {k: v.shape for k, v in images.items()} }")

joints:
  Rotation: -0.0000
  Pitch: 0.0169
  Elbow: 0.0076
  Wrist_Pitch: 0.0007
  Wrist_Roll: 0.0000
  Jaw: -0.0000
images: {'default': (480, 640, 3)}


## Camera Observations as Numpy Arrays

`get_observation()` returns raw numpy arrays — useful for policies and direct processing.

In [ ]:
# Camera images from observations are raw numpy uint8 arrays
obs = sim.get_observation("so100")
for k, v in obs.items():
    if hasattr(v, 'shape') and getattr(v, 'ndim', 0) >= 2:
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}, range=[{v.min()}, {v.max()}]")

        # Display with matplotlib
        try:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(6, 4))
            plt.imshow(v)
            plt.title(f"observation['{k}']")
            plt.axis("off")
            plt.show()
        except ImportError:
            print("  (install matplotlib for inline display)")

## Sending Actions

In [15]:
import numpy as np

action = {k: float(np.sin(0.5)) for k in joints}
sim.send_action(action, robot_name="so100")

for _ in range(50):
    sim.step(n_steps=1)

new_obs = sim.get_observation("so100")
for k in joints:
    actual = new_obs.get(k, 0.0)
    print(f"  {k}: target={action[k]:.4f} → actual={actual:.4f}")

  Rotation: target=0.4794 → actual=0.0928
  Pitch: target=0.4794 → actual=0.0570
  Elbow: target=0.4794 → actual=0.1304
  Wrist_Pitch: target=0.4794 → actual=0.1501
  Wrist_Roll: target=0.4794 → actual=0.1810
  Jaw: target=0.4794 → actual=0.1794


## Rendering

Two ways to get images:
1. `sim.render()` → returns base64 PNG (for AgentTool / Strands Agent)
2. `sim.get_observation()` → returns raw numpy arrays (for policies + display)

We'll use a helper to display both in the notebook.

In [ ]:
import numpy as np
from IPython.display import display, Image as IPImage
import io

def show_sim(sim, camera="default", width=640, height=480):
    """Render simulation and display inline in notebook."""
    result = sim.render(camera_name=camera, width=width, height=height)

    if result.get("status") != "success":
        print(result["content"][0]["text"])
        return None

    # render() returns PNG bytes in content[1]["image"]["source"]["bytes"]
    for item in result.get("content", []):
        if "image" in item:
            png_bytes = item["image"]["source"]["bytes"]
            display(IPImage(data=png_bytes))
            print(result["content"][0]["text"])
            return png_bytes

    # Fallback: try matplotlib with observation numpy arrays
    obs = sim.get_observation(list(sim._world.robots.keys())[0] if sim._world else None)
    for k, v in obs.items():
        if hasattr(v, 'shape') and getattr(v, 'ndim', 0) >= 2:
            try:
                import matplotlib.pyplot as plt
                plt.figure(figsize=(8, 6))
                plt.imshow(v)
                plt.title(f"Camera: {k}")
                plt.axis("off")
                plt.show()
                return v
            except ImportError:
                pass

    print("No image available (headless without EGL/OSMesa)")
    return None

In [ ]:
# Render and display inline
show_sim(sim, width=640, height=480)

## Physics Features (PhysicsMixin)

In [17]:
# Save/restore state checkpoints
sim.save_state("checkpoint_1")
sim.step(n_steps=200)
sim.load_state("checkpoint_1")
print("state restored ✅")

state restored ✅


In [18]:
sim.destroy()
print("world destroyed ✅")

world destroyed ✅
